1. Basics of Socket Programming

A socket is the OS-level software abstraction of a network connection. To the operating system, a socket is just a file descriptor—a pointer to a data stream. To create one, we must explicitly define the addressing scheme and the transport protocol.

In [16]:
import socket

"""
Creating a TCP Socket:
- socket.AF_INET: Specifies the Address Family as IPv4 (e.g., 128.173.235.1). 
  If we wanted IPv6, we would use socket.AF_INET6.
- socket.SOCK_STREAM: Specifies a continuous, reliable, two-way byte stream. 
  The OS maps this directly to the Transmission Control Protocol (TCP).
"""
tcp_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
print(f"TCP Socket Created: {tcp_socket}")

"""
Creating a UDP Socket:
- socket.SOCK_DGRAM: Specifies discrete datagrams (messages). 
  This maps to the User Datagram Protocol (UDP), which is connectionless and unreliable, 
  meaning packets might arrive out of order or drop entirely without warning.
"""
udp_socket = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
print(f"UDP Socket Created: {udp_socket}")

# Always close sockets to free up the file descriptors in the OS kernel.
tcp_socket.close()
udp_socket.close()

TCP Socket Created: <socket.socket fd=89, family=2, type=1, proto=0, laddr=('0.0.0.0', 0)>
UDP Socket Created: <socket.socket fd=90, family=2, type=2, proto=0, laddr=('0.0.0.0', 0)>


2. UDP Server and Client

UDP is connectionless. There is no 3-way handshake, no connection state, and no teardown. The server simply binds to a port and waits for incoming datagrams (recvfrom). The client sends data directly to an IP and port (sendto) without verifying if the server is even online.

Note: We run servers in background threads in this notebook so blocking calls do not freeze your Jupyter kernel.

In [17]:
import threading
import time

UDP_IP = "127.0.0.1" # Localhost loopback interface
UDP_PORT = 5001      # Arbitrary unprivileged port (> 1023)

def run_udp_server():
    server = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    
    # bind() tells the OS: "Route all incoming UDP traffic on port 5001 to this program."
    server.bind((UDP_IP, UDP_PORT))
    print(f"[UDP SERVER] Listening on {UDP_PORT}...")
    
    # recvfrom() blocks until a packet arrives. 
    # 1024 is the buffer size (maximum bytes to read at once).
    # It returns a tuple: the raw byte data, and the sender's (IP, Port) address.
    # We MUST know the sender's address to reply, since UDP has no persistent connection.
    data, addr = server.recvfrom(1024)
    print(f"[UDP SERVER] Received '{data.decode()}' from {addr}")
    
    # Send a reply back to the exact address we just captured.
    server.sendto(b"Hokie Bird ACK!", addr)
    server.close()

# Start server in the background so Jupyter remains responsive
threading.Thread(target=run_udp_server).start()
time.sleep(0.5) 

# --- UDP CLIENT ---
client = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
print("[UDP CLIENT] Sending payload...")

# sendto() requires the payload (in bytes, hence the 'b' prefix) and a destination tuple.
# Because UDP is fire-and-forget, this function succeeds immediately even if the server is offline.
client.sendto(b"Hello UDP via Jupyter!", (UDP_IP, UDP_PORT))

response, server_addr = client.recvfrom(1024)
print(f"[UDP CLIENT] Server replied: {response.decode()}")
client.close()

[UDP SERVER] Listening on 5001...
[UDP CLIENT] Sending payload...
[UDP SERVER] Received 'Hello UDP via Jupyter!' from ('127.0.0.1', 55642)
[UDP CLIENT] Server replied: Hokie Bird ACK!


3. TCP Server and Client

TCP requires a strict state machine. The server must bind(), listen(), and accept() to establish a connection. The client must connect() to complete the 3-way handshake. Only then can data flow over the dedicated connection using recv() and sendall().

In [6]:
TCP_IP = "127.0.0.1"
TCP_PORT = 5002

def run_tcp_server():
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    
    # SO_REUSEADDR allows us to immediately reuse the port if the server restarts.
    # Without this, the OS keeps the port locked in a TIME_WAIT state for several minutes.
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    
    server.bind((TCP_IP, TCP_PORT))
    
    # listen(1) tells the OS to start accepting the TCP 3-way handshake.
    # '1' is the backlog: the OS will queue up to 1 pending connection while our code is busy.
    server.listen(1)
    print(f"[TCP SERVER] Listening on {TCP_PORT}...")
    
    # accept() blocks the thread until a client finishes the handshake.
    # Crucially, it returns a BRAND NEW socket object ('conn') dedicated solely to this client.
    # The original 'server' socket continues listening for other new clients.
    conn, addr = server.accept()
    print(f"[TCP SERVER] Connection established with {addr}")
    
    data = conn.recv(1024)
    
    # sendall() ensures all bytes are transmitted, handling partial sends under the hood.
    conn.sendall(b"TCP Data Received: " + data)
    
    # Both the client-specific socket and the main listening socket must be closed.
    conn.close()
    server.close()

threading.Thread(target=run_tcp_server).start()
time.sleep(0.5)

# --- TCP CLIENT ---
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
print("[TCP CLIENT] Connecting to server...")

# connect() actively initiates the SYN -> SYN-ACK -> ACK TCP handshake.
# If the server is offline, this will throw a ConnectionRefusedError.
client.connect((TCP_IP, TCP_PORT))

client.sendall(b"Hello TCP via Jupyter!")
response = client.recv(1024)
print(f"[TCP CLIENT] Server replied: {response.decode()}")
client.close()

[TCP SERVER] Listening on 5002...
[TCP CLIENT] Connecting to server...
[TCP SERVER] Connection established with ('127.0.0.1', 51665)
[TCP CLIENT] Server replied: TCP Data Received: Hello TCP via Jupyter!


4. Socket Timeouts

Networks are inherently unreliable. If a server goes offline silently, or a router drops your packets, your client's recv() or connect() call will hang indefinitely waiting for an answer. Applying a timeout ensures your application can recover, log an error, and continue running.

In [7]:
# 198.51.100.1 is a reserved TEST-NET IP configured to drop all traffic (a blackhole).
BLACKHOLE_IP = "198.51.100.1" 
PORT = 80

client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# Force the socket to abandon any operation that takes longer than 1.5 seconds.
# This changes the socket from "blocking" to "timeout" mode.
client.settimeout(1.5)

print(f"[TIMEOUT] Attempting to connect to {BLACKHOLE_IP}...")
try:
    # Because it's a blackhole, the TCP SYN packet gets no SYN-ACK reply.
    # After 1.5 seconds, Python intervenes and raises an exception.
    client.connect((BLACKHOLE_IP, PORT))
except socket.timeout:
    print("[TIMEOUT] Error: The connection timed out! The network link might be down.")
except Exception as e:
    print(f"[TIMEOUT] Unexpected error: {e}")
finally:
    # The finally block guarantees the socket is cleaned up, even if an exception occurs.
    client.close()

[TIMEOUT] Attempting to connect to 198.51.100.1...
[TIMEOUT] Error: The connection timed out! The network link might be down.


5. Waiting for multiple events (multiplexing)

Real-world servers handle thousands of clients simultaneously. If a server calls accept() or recv() on a standard blocking socket, the entire application freezes waiting for that specific client, ignoring everyone else.

To solve this without spawning thousands of expensive threads, we use Multiplexing via the selectors module (which maps to OS-level system calls like epoll or select). The server registers all active sockets with the OS. The OS then monitors them in the background and wakes the server up only when a socket is ready to be read from or written to.

In [8]:
import socket
import selectors
import threading
import time

# Create a default selector based on the operating system
sel = selectors.DefaultSelector()
HOST = "127.0.0.1"
PORT = 5005

def run_multiplexing_server():
    server_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server_sock.bind((HOST, PORT))
    server_sock.listen()
    
    # CRITICAL: Put the listening socket in non-blocking mode.
    # It will never freeze the program while waiting for a connection.
    server_sock.setblocking(False)
    
    # Register the listening socket with the selector.
    # data="LISTENING" acts as a tag so we can identify this socket later.
    sel.register(server_sock, selectors.EVENT_READ, data="LISTENING")
    print(f"[SERVER] Single-threaded multiplexing server started on {PORT}...")

    # Run the event loop for 3 seconds for this notebook demonstration
    end_time = time.time() + 3.0
    while time.time() < end_time:
        # sel.select() pauses execution here until the OS reports a socket is ready.
        events = sel.select(timeout=0.5)
        
        for key, mask in events:
            sock = key.fileobj # The actual socket object that is ready
            
            # Scenario A: The LISTENING socket is ready. This means a new client is connecting!
            if key.data == "LISTENING":
                # We can safely call accept() knowing it will return immediately.
                conn, addr = sock.accept()
                print(f"[SERVER] Accepted new connection from {addr}")
                
                # Put the NEW client socket in non-blocking mode too
                conn.setblocking(False)
                
                # Register the NEW client socket with the selector.
                # Tag it as "CLIENT" so we know it's meant for reading data, not accepting connections.
                sel.register(conn, selectors.EVENT_READ, data="CLIENT")
                
            # Scenario B: A CLIENT socket is ready. This means they sent us data (or disconnected).
            elif key.data == "CLIENT":
                try:
                    data = sock.recv(1024)
                    if data:
                        print(f"[SERVER] Echoing data back: {data.decode()}")
                        sock.sendall(data)
                    else:
                        # An empty recv() means the client closed the connection.
                        print(f"[SERVER] Client disconnected.")
                        sel.unregister(sock) # Stop monitoring this socket
                        sock.close()
                except ConnectionResetError:
                    print(f"[SERVER] Client forcefully disconnected.")
                    sel.unregister(sock)
                    sock.close()

    # Cleanup the main server socket after the demo loop finishes
    sel.unregister(server_sock)
    server_sock.close()
    print("[SERVER] Demonstration finished. Server closed.")

# 1. Start the multiplexing server in the background
threading.Thread(target=run_multiplexing_server).start()
time.sleep(0.5)

# 2. Simulate three separate clients connecting simultaneously
def simulate_client(client_id):
    client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client.connect((HOST, PORT))
    
    message = f"Hello from Client {client_id}!"
    client.sendall(message.encode())
    
    response = client.recv(1024)
    print(f"[CLIENT {client_id}] Received: {response.decode()}")
    client.close()

# Launch the three clients rapidly
for i in range(1, 4):
    threading.Thread(target=simulate_client, args=(i,)).start()

[SERVER] Single-threaded multiplexing server started on 5005...
[SERVER] Accepted new connection from ('127.0.0.1', 51676)
[SERVER] Accepted new connection from ('127.0.0.1', 51677)
[SERVER] Echoing data back: Hello from Client 1!
[CLIENT 1] Received: Hello from Client 1!
[SERVER] Accepted new connection from ('127.0.0.1', 51678)
[SERVER] Echoing data back: Hello from Client 2!
[CLIENT 2] Received: Hello from Client 2!
[SERVER] Client disconnected.
[SERVER] Client disconnected.
[SERVER] Echoing data back: Hello from Client 3!
[CLIENT 3] Received: Hello from Client 3!
[SERVER] Client disconnected.


[SERVER] Demonstration finished. Server closed.


6. Endianness (Byte Order)

When your computer stores a multi-byte number in memory, it has to decide which byte goes first. Most modern CPUs (Intel/AMD) use Little-Endian (least significant byte first). However, the internet standardizes on Big-Endian (most significant byte first), also known as Network Byte Order. If we do not convert our numbers before sending them over a socket, the receiving computer might read them backward.

In [9]:
import sys
import socket

print(f"1. Your CPU's native byte order is: {sys.byteorder}")

# Let's take a standard port number
original_port = 80
print(f"2. Original Port Number: {original_port}")

# Convert it to Network Byte Order (Big-Endian) using 'htons' (Host to Network Short)
# 'Short' refers to a 16-bit integer.
network_port = socket.htons(original_port)
print(f"3. Port in Network Byte Order (Raw Integer): {network_port}")

# Convert it back to Host Byte Order using 'ntohs' (Network to Host Short)
restored_port = socket.ntohs(network_port)
print(f"4. Restored to Host Byte Order: {restored_port}")

# Proof of what the bytes actually look like in memory:
# Let's use a larger number: 1024 (which is 0x0400 in Hex)
num = 1024
print(f"\nHex representation of 1024: 0x0400")
print(f"Little-Endian bytes: {num.to_bytes(2, byteorder='little')}")
print(f"Big-Endian (Network) bytes: {num.to_bytes(2, byteorder='big')}")

1. Your CPU's native byte order is: little
2. Original Port Number: 80
3. Port in Network Byte Order (Raw Integer): 20480
4. Restored to Host Byte Order: 80

Hex representation of 1024: 0x0400
Little-Endian bytes: b'\x00\x04'
Big-Endian (Network) bytes: b'\x04\x00'


7. TCP framing

TCP is a continuous stream of bytes, not a stream of discrete messages. 

TCP Framing Problem:
No Message Boundaries: TCP guarantees that data arrives in order and without corruption, but it completely ignores application-level message boundaries.

Fragmentation and Merging: A single send() call from an application might be split into multiple pieces by the network stack (arriving as multiple recv() calls), or multiple send() calls might be mashed together into a single recv() call.

Buffer Guessing Fails: If an app tries to read a fixed number of bytes, it risks cutting a message in half or reading part of the next message, leading to corrupted or out-of-sync data

Solution: 

To solve the TCP framing problem, applications must build a custom framing layer on top of the TCP stream.

Length Prefixing: Prepend a fixed-size header (e.g., 4 bytes) to every message that specifies how many bytes the incoming message will be. The receiver reads the prefix first, then reads exactly that number of bytes

Fixed-Size Messages: Make every message the exact same length so the receiver always knows precisely how many bytes to pull from the stream.


Question: Does UDP have the framing problem?

In [10]:
import struct

# 1. Prepare the payload
payload = "Let's Go Hokies!"
# Strings must be encoded into raw bytes before transmission
encoded_payload = payload.encode('utf-8')
payload_length = len(encoded_payload)

print(f"Payload: '{payload}' (Length: {payload_length} bytes)")

"""
2. Pack the Header using struct.pack()
Format String Breakdown: '!I'
- '!' forces Network Byte Order (Big-Endian).
- 'I' specifies an unsigned 32-bit integer (which always takes exactly 4 bytes).
"""
binary_header = struct.pack('!I', payload_length)
print(f"Packed 4-byte header: {binary_header}")

# 3. Combine header and payload to create the final packet for transmission
packet_to_send = binary_header + encoded_payload
print(f"Complete packet on the wire: {packet_to_send}\n")

"""
4. Simulating the Receiver
The receiver ALWAYS reads exactly 4 bytes first to decode the header.
"""
received_header = packet_to_send[:4] 
# unpack always returns a tuple, so we grab the first element [0]
expected_length = struct.unpack('!I', received_header)[0]

print(f"Receiver decoded header. Expecting a {expected_length}-byte payload...")

# Now the receiver knows exactly how many bytes to read for the payload
received_payload_bytes = packet_to_send[4:4+expected_length]
final_message = received_payload_bytes.decode('utf-8')

print(f"Receiver successfully extracted: {final_message}")

Payload: 'Let's Go Hokies!' (Length: 16 bytes)
Packed 4-byte header: b'\x00\x00\x00\x10'
Complete packet on the wire: b"\x00\x00\x00\x10Let's Go Hokies!"

Receiver decoded header. Expecting a 16-byte payload...
Receiver successfully extracted: Let's Go Hokies!


8. Application-layer serialization (JSON)

Raw strings are rarely enough for modern applications. We usually need to send structured data (like dictionaries or objects). We serialize this data into JSON strings, encode the JSON to UTF-8 bytes, and then apply our binary length header from the previous step.

In [11]:
import json
import struct

# 1. A complex Python dictionary (Object)
sensor_data = {
    "device_id": "sensor_01",
    "temperature_c": 22.5,
    "active": True
}

# 2. Serialize Dictionary -> JSON String
json_string = json.dumps(sensor_data)
print(f"Serialized JSON string: {json_string}")

# 3. Encode JSON String -> UTF-8 Bytes
json_bytes = json_string.encode('utf-8')

# 4. Apply Binary Framing Header
packet_length = len(json_bytes)
header = struct.pack('!I', packet_length)
final_packet = header + json_bytes

print(f"Ready for socket.send(): {final_packet}\n")

# --- SIMULATING THE RECEIVER ---
extracted_length = struct.unpack('!I', final_packet[:4])[0]
extracted_json_bytes = final_packet[4:4+extracted_length]

# 5. Decode Bytes -> JSON String -> Python Dictionary
restored_dict = json.loads(extracted_json_bytes.decode('utf-8'))
print(f"Restored Python dictionary: {restored_dict}")
print(f"Accessing data: Temperature is {restored_dict['temperature_c']}°C")

Serialized JSON string: {"device_id": "sensor_01", "temperature_c": 22.5, "active": true}
Ready for socket.send(): b'\x00\x00\x00A{"device_id": "sensor_01", "temperature_c": 22.5, "active": true}'

Restored Python dictionary: {'device_id': 'sensor_01', 'temperature_c': 22.5, 'active': True}
Accessing data: Temperature is 22.5°C
